# Feature engineering for neighborhood segmentation

This notebook merges census, crime, business licence, and building permit
data into a single community-level feature matrix, standardises the
features, handles missing values, and applies PCA for dimensionality
reduction.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, '..')
from src.data_loader import (
    load_all_datasets, build_feature_matrix, FEATURE_COLUMNS
)
from src.model import fit_pca, get_pca_loadings

pd.set_option('display.max_columns', 50)

## 1. Load the four source datasets

Each dataset comes from the Calgary Open Data portal and is cached
locally after the first download.

In [ ]:
datasets = load_all_datasets(force_refresh=False)

for name, data in datasets.items():
    print(f"{name}: {data.shape[0]} rows, {data.shape[1]} columns")

## 2. Build the unified feature matrix

The `build_feature_matrix` function aggregates each dataset to the
community level and merges them on community name. It also imputes
missing values with the median and standardises all features using
z-score scaling.

In [ ]:
raw_df, scaled_df, scaler = build_feature_matrix(datasets=datasets)

print(f"Communities: {len(raw_df)}")
print(f"Feature columns: {FEATURE_COLUMNS}")
print(f"\nRaw feature matrix:")
raw_df.head(10)

## 3. Missing values before imputation

Let us check how many communities had missing data for each indicator
before the median imputation step filled them in.

In [ ]:
# Rebuild the merged frame without imputation to inspect missingness
from src.data_loader import (
    _build_census_features, _build_crime_features,
    _build_business_features, _build_permit_features
)

census_feat = _build_census_features(datasets['census'])
crime_feat = _build_crime_features(datasets['crime'])
business_feat = _build_business_features(datasets['business'])
permit_feat = _build_permit_features(datasets['permits'])

merged = census_feat.copy()
for right in [crime_feat, business_feat, permit_feat]:
    merged = merged.merge(right, on='community', how='outer')

merged['crime_rate'] = np.where(
    merged['total_population'] > 0,
    merged['total_crimes'] / merged['total_population'] * 1000,
    np.nan
)

for col in FEATURE_COLUMNS:
    if col not in merged.columns:
        merged[col] = np.nan

missing_pct = merged[FEATURE_COLUMNS].isna().mean() * 100
print("Missing percentage per feature (before imputation):")
print(missing_pct.round(1).to_string())

## 4. Standardised feature distributions

After z-score standardisation, each feature has mean 0 and standard
deviation 1. This is important for distance-based methods like KMeans.

In [ ]:
print("Scaled feature summary:")
print(scaled_df[FEATURE_COLUMNS].describe().round(3))

In [ ]:
n_cols = 3
n_rows_plot = (len(FEATURE_COLUMNS) + n_cols - 1) // n_cols

fig = make_subplots(rows=n_rows_plot, cols=n_cols,
                    subplot_titles=FEATURE_COLUMNS)

for i, col in enumerate(FEATURE_COLUMNS):
    r, c = divmod(i, n_cols)
    fig.add_trace(
        go.Histogram(x=scaled_df[col], nbinsx=30, name=col),
        row=r + 1, col=c + 1
    )

fig.update_layout(height=200 * n_rows_plot,
                  title_text='Scaled feature distributions',
                  showlegend=False)
fig.show()

## 5. Feature correlations

In [ ]:
corr = raw_df[FEATURE_COLUMNS].corr()

fig = px.imshow(
    corr, text_auto='.2f', color_continuous_scale='RdBu_r',
    title='Feature correlation matrix (original scale)',
    aspect='auto'
)
fig.update_layout(height=550, width=650)
fig.show()

## 6. PCA for dimensionality reduction

Principal Component Analysis identifies the directions of maximum
variance. We first fit PCA on the full set of components, then look
at how much variance each component explains to decide how many to
retain.

In [ ]:
X_scaled = scaled_df[FEATURE_COLUMNS].values
pca, X_pca = fit_pca(X_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print("Explained variance per component:")
for i, (ev, cum) in enumerate(zip(explained, cumulative)):
    print(f"  PC{i+1}: {ev:.3f} (cumulative: {cum:.3f})")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=[f'PC{i+1}' for i in range(len(explained))],
    y=explained, name='Individual'
))
fig.add_trace(go.Scatter(
    x=[f'PC{i+1}' for i in range(len(cumulative))],
    y=cumulative, mode='lines+markers', name='Cumulative'
))
fig.add_hline(y=0.90, line_dash='dash', line_color='red',
              annotation_text='90% threshold')
fig.update_layout(
    title='PCA explained variance',
    xaxis_title='Principal component',
    yaxis_title='Explained variance ratio',
    height=400
)
fig.show()

## 7. PCA loadings

The loadings show which original features contribute most to each
principal component, helping us interpret what each component captures.

In [ ]:
loadings = get_pca_loadings(pca, FEATURE_COLUMNS)
print("PCA loadings (first 4 components):")
print(loadings.iloc[:4].T.round(3).to_string())

In [ ]:
# Scatter plot of communities in PC1-PC2 space
pca_plot_df = pd.DataFrame({
    'community': scaled_df['community'],
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
})

fig = px.scatter(
    pca_plot_df, x='PC1', y='PC2',
    hover_data=['community'],
    title='Communities in PC1 vs PC2 space',
    labels={'PC1': f'PC1 ({explained[0]:.1%} variance)',
            'PC2': f'PC2 ({explained[1]:.1%} variance)'},
    opacity=0.6
)
fig.update_layout(height=500)
fig.show()

## 8. Summary

The feature engineering pipeline produces a matrix of 10 indicators per
community, drawn from four city datasets:

| Source | Features |
|---|---|
| Census | total_population, median_age_proxy, gender_ratio |
| Crime | total_crimes, crime_rate |
| Business licences | business_count, business_diversity |
| Building permits | avg_building_cost, permit_count, housing_mix |

Missing values were imputed with medians and all features were
standardised. PCA shows that the first few components capture most
of the variance, confirming that there is meaningful structure for
clustering to discover. The next notebook applies KMeans and
Agglomerative clustering to identify distinct neighborhood segments.